In [ ]:
##### Dicom file processing and image enhancement for 2D X-ray images ####

import os
import glob
import pydicom
import numpy as np
import cv2
from skimage import exposure, measure
import matplotlib.pyplot as plt

input_dir = r"C:\Users\csm02\Desktop\edward\bmd\1 src\dataset-dcm\train\gather"
output_dir = r"C:\Users\csm02\Desktop\edward\bmd\1 src\dataset-dcm\train\gather_output"
os.makedirs(output_dir, exist_ok=True)

# 42 files in total, but we will loop through all to ensure robustness
dicom_files = glob.glob(os.path.join(input_dir, "*"))

print(f"{len(dicom_files)} 2D Dicom File Found. Starting processing...")

def load_dicom(path):
    dcm = pydicom.dcmread(path, force=True)
    arr = dcm.pixel_array.astype(np.float64)
    arr = arr * float(getattr(dcm,'RescaleSlope',1.0)) + float(getattr(dcm,'RescaleIntercept',0.0))
    if str(getattr(dcm,'PhotometricInterpretation','MONOCHROME2')).strip()=='MONOCHROME1':
        arr = arr.max() - arr
    ps = list(getattr(dcm,'PixelSpacing',[0.1,0.1]))
    meta = {
        'patient_id'   : str(getattr(dcm,'PatientID',   'Unknown')),
        'patient_age'  : str(getattr(dcm,'PatientAge',  'Unknown')),
        'patient_sex'  : str(getattr(dcm,'PatientSex',  'Unknown')),
        'modality'     : str(getattr(dcm,'Modality',    'CR')),
        'study_date'   : str(getattr(dcm,'StudyDate',   '')),
        'pixel_spacing': ps,
        'shape'        : arr.shape,
        'filename'     : os.path.basename(path),
        'kvp'          : str(getattr(dcm,'KVP',         'Unknown')),
    }
    return arr, meta, dcm


def preprocess_xray(arr, size=256, clip=0.04):
    h, w   = arr.shape
    mx     = max(h, w)
    pad    = np.zeros((mx,mx), dtype=arr.dtype)
    yo,xo  = (mx-h)//2, (mx-w)//2
    pad[yo:yo+h, xo:xo+w] = arr
    resized = cv2.resize(pad,(size,size), interpolation=cv2.INTER_LINEAR)
    vmin,vmax = np.percentile(resized,[1,99])
    norm  = np.clip((resized-vmin)/(vmax-vmin+1e-8),0,1).astype(np.float32)
    clahe = exposure.equalize_adapthist(norm, clip_limit=clip).astype(np.float32)
    return norm, clahe, {'yo':yo,'xo':xo,'orig_h':h,'orig_w':w,'mx':mx}

for filepath in dicom_files:
    if os.path.isdir(filepath):
        continue
        
    try:        
        arr, meta, dcm = load_dicom(filepath)

        # 1. bone contour enhancement: 
        # Resize 256 & CHACHE(clip_limit=0.04)
        norm_img, clahe_img, meta = preprocess_xray(arr,  clip=0.04)
        
        # 2. Convert to uint8 for saving
        # 2. Convert 0.0~1.0 to 0~255 uint8 for saving
        img_for_save = (clahe_img * 255.0).astype(np.uint8)
        
        # 3. Filename generation (original file name + "_clahe.png")
        file_root, _ = os.path.splitext(os.path.basename(filepath))
        output_filename = f"{file_root}_clahe.png"              

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(clahe_img, cmap="gray")
        ax.axis("off")
        plt.savefig(os.path.join(output_dir, output_filename), bbox_inches='tight', dpi=150)
        
        #plt.show()
        plt.close(fig)

        print(f"✅ Outline sharpening completed (saved): {output_filename}")
        
    except Exception as e:
        print(f"❌ Processing failed ({os.path.basename(filepath)}): {e}")
print("All 2D data loops completed!")

총 42개의 2D X-ray 파일 전처리를 시작합니다.
✅ 윤곽선 선명화 완료 (저장): 1.2.826.0.1.3680043.8.498.10406657579016312201439626146133412368-c_clahe.png
✅ 윤곽선 선명화 완료 (저장): 1.2.826.0.1.3680043.8.498.10947130043296900260026400505615377321-c_clahe.png
✅ 윤곽선 선명화 완료 (저장): 1.2.826.0.1.3680043.8.498.11121020226184051416110461667223845149-c_clahe.png
✅ 윤곽선 선명화 완료 (저장): 1.2.826.0.1.3680043.8.498.11207126462590105123217163368073993344-c_clahe.png
✅ 윤곽선 선명화 완료 (저장): 1.2.826.0.1.3680043.8.498.11227155170490832168629184106125649390-c_clahe.png
✅ 윤곽선 선명화 완료 (저장): 1.2.826.0.1.3680043.8.498.11430185427783861938088406669987177226-c_clahe.png
✅ 윤곽선 선명화 완료 (저장): 1.2.826.0.1.3680043.8.498.11479964994316715381867899205585900227-c_clahe.png
✅ 윤곽선 선명화 완료 (저장): 1.2.826.0.1.3680043.8.498.11666143137939211262402653035688444502-c_clahe.png
✅ 윤곽선 선명화 완료 (저장): 1.2.826.0.1.3680043.8.498.12209232711532162247608981043714969492-c_clahe.png
✅ 윤곽선 선명화 완료 (저장): 1.2.826.0.1.3680043.8.498.12608809850577365286275480029864466891-c_clahe.png
✅ 윤곽선 선명화

In [1]:
! pip install torch torchvision pydicom opencv-python scikit-image matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import glob
import numpy as np
import pydicom
import cv2
import matplotlib.pyplot as plt
from skimage import exposure

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

# GPU 가속 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")

# =========================================================================
# 1. 의료 영상 전처리 함수 (기존에 검증된 함수 활용)
# =========================================================================
def load_dicom(path):
    dcm = pydicom.dcmread(path, force=True)
    arr = dcm.pixel_array.astype(np.float64)
    arr = arr * float(getattr(dcm,'RescaleSlope',1.0)) + float(getattr(dcm,'RescaleIntercept',0.0))
    if str(getattr(dcm,'PhotometricInterpretation','MONOCHROME2')).strip()=='MONOCHROME1':
        arr = arr.max() - arr
    return arr

def preprocess_xray(arr, size=256, clip=0.04):
    h, w   = arr.shape
    mx     = max(h, w)
    pad    = np.zeros((mx,mx), dtype=arr.dtype)
    yo,xo  = (mx-h)//2, (mx-w)//2
    pad[yo:yo+h, xo:xo+w] = arr
    resized = cv2.resize(pad,(size,size), interpolation=cv2.INTER_LINEAR)
    vmin,vmax = np.percentile(resized,[1,99])
    norm  = np.clip((resized-vmin)/(vmax-vmin+1e-8),0,1).astype(np.float32)
    clahe = exposure.equalize_adapthist(norm, clip_limit=clip).astype(np.float32)
    return clahe, {'yo':yo, 'xo':xo, 'mx':mx}

# =========================================================================
# 2. PyTorch 커스텀 데이터셋 정의 (DICOM Image & PNG Mask 매핑)
# =========================================================================
class SpineDataset(Dataset):
    def __init__(self, dicom_dir, mask_dir, size=256):
        self.dicom_paths = sorted(glob.glob(os.path.join(dicom_dir, "*")))
        self.mask_dir = mask_dir
        self.size = size
        
        # 디렉토리 스크리닝 (폴더는 제외)
        self.dicom_paths = [p for p in self.dicom_paths if os.path.isfile(p)]

    def __len__(self):
        return len(self.dicom_paths)

    def __getitem__(self, idx):
        dcm_path = self.dicom_paths[idx]
        filename = os.path.basename(dcm_path)
        file_root, _ = os.path.splitext(filename)
        
        # 1. DICOM 로드 및 전처리 (CLAHE 출력 적용)
        raw_arr = load_dicom(dcm_path)
        clahe_img, _ = preprocess_xray(raw_arr, size=self.size)
        
        # 2. 정답 마스크 로드 (SegmentationClass 폴더 안에서 짝을 찾음)
        # CVAT 익스포트 포맷에 따라 확장자가 .png로 저장됩니다.
        mask_path = os.path.join(self.mask_dir, f"{file_root}.png")
        
        if os.path.exists(mask_path):
            # 마스크는 Grayscale로 읽은 후 이미지와 동일한 size로 리사이즈
            mask_arr = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask_resized = cv2.resize(mask_arr, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
            # L4 라벨 영역(0이 아닌 값)을 1로, 배경을 0으로 이진화
            mask = (mask_resized > 0).astype(np.float32)
        else:
            # 마스크 파일이 매핑되지 않을 경우 빈 마스크 생성 (에러 방지)
            mask = np.zeros((self.size, self.size), dtype=np.float32)

        # PyTorch 텐서 변환 (Channel, Height, Width) -> (1, 256, 256)
        image_tensor = torch.tensor(clahe_img).unsqueeze(0).float()
        mask_tensor = torch.tensor(mask).unsqueeze(0).float()

        return image_tensor, mask_tensor

# =========================================================================
# 3. 가벼운 2D U-Net 아키텍처 정의
# =========================================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.inc   = DoubleConv(1, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        
        self.up1   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)
        self.up2   = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv2 = DoubleConv(64, 32)
        self.outc  = nn.Conv2d(32, 1, 1) # 이진 분류 (L4 뼈인가 아닌가)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        
        x = self.up1(x3)
        x = torch.cat([x, x2], dim=1)
        x = self.conv1(x)
        
        x = self.up2(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv2(x)
        return self.outc(x)

# =========================================================================
# 4. 메인 학습 실행 (Training Loop)
# =========================================================================
# 경로 설정
dicom_dir = r"C:\Users\csm02\Desktop\edward\bmd\1 src\dataset-dcm\train\gather"
mask_dir  = r"C:\Users\csm02\Desktop\edward\bmd\1 src\dataset-dcm\train\gather_mask\SegmentationClass"

# 데이터 로더 구성
dataset = SpineDataset(dicom_dir=dicom_dir, mask_dir=mask_dir, size=256)

# 42장 중 34장은 학습(Train), 8장은 검증(Validation)용으로 분할
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# 모델, 손실함수, 옵티마이저 선언
model = UNet2D().to(device)
criterion = nn.BCEWithLogitsLoss() # 이진 세그멘테이션 표준 손실함수
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(f"학습 데이터: {train_size}장, 검증 데이터: {val_size}장")
print("🚀 학습을 시작합니다.")

epochs = 20
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    train_loss /= len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss:.4f}")

# 학습된 가중치 저장
torch.save(model.state_dict(), "unet_l4_spine.pth")
print("💾 모델 가중치 저장 완료 (unet_l4_spine.pth)")

# =========================================================================
# 5. 테스트 및 시각화 (Inference & Test)
# =========================================================================
print("\n🔍 검증 데이터셋 테스트 및 시각화를 수행합니다...")
model.eval()

with torch.no_grad():
    # 검증 세트에서 3개만 뽑아서 시각화 확인
    for idx, (image, mask) in enumerate(val_loader):
        if idx >= 3: break 
        
        image_dev, mask_dev = image.to(device), mask.to(device)
        output = model(image_dev)
        
        # 시그모이드를 거쳐 0.5 이상인 확률 영역을 1(마스크)로 판단
        pred_mask = (torch.sigmoid(output) > 0.5).float()
        
        # 텐서를 넘파이 배열로 복원 (시각화용)
        img_np = image.squeeze().cpu().numpy()
        gt_np = mask.squeeze().cpu().numpy()
        pred_np = pred_mask.squeeze().cpu().numpy()
        
        # 결과 플롯 창 그리기
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 3, 1)
        plt.imshow(img_np, cmap='gray')
        plt.title("Original X-ray (CLAHE)")
        plt.axis('off')
        
        plt.subplot(1, 3, 2)
        plt.imshow(gt_np, cmap='jet', alpha=0.7)
        plt.title("Ground Truth (CVAT)")
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        plt.imshow(img_np, cmap='gray')
        plt.imshow(pred_np, cmap='red_fading', alpha=0.4 if np.max(pred_np)>0 else 0) # 예측된 L4 영역 오버레이
        plt.imshow(pred_np, cmap='Reds', alpha=0.5)
        plt.title("AI Predicted L4 Mask")
        plt.axis('off')
        
        plt.show()

현재 사용 중인 장치: cuda


c:\Users\csm02\.conda\envs\bmd\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


학습 데이터: 33장, 검증 데이터: 9장
🚀 학습을 시작합니다.


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
